# Study 914 — Securities-Lending Offset — the teardown

The estimand, the five-pair table, the structural SPY/IVV control and its confounds, the two-sided fee-history sweep that decides the residual's sign, the pooled residuals, the era cut, the frequency check, the borrow-swept pair trade, the power arithmetic, and the live synthetic control. Every real number is frozen from `docs/results.md` (Fingerprint `d080dc025da8`, as-of 2026-06-30).

## The estimand

For funds $a$, $b$ in the same asset class, on month-end log total returns:

$$\text{residual}(a,b) \;=\; 12\cdot\overline{\left(\log r_{a,t} - \log r_{b,t}\right)} \;+\; (\mathrm{ER}_a - \mathrm{ER}_b)$$

Under the null of *no differential lending passthrough and perfect replication*, the residual is zero. The lending-offset claim predicts a positive residual for the leg whose lending programme returns more.

Three design points, stated once:

1. **No signal, so no execution lag — and none is faked.** The residual is an accounting identity on contemporaneous returns. The *tradable* leg is a **static** pair of two fixed tickers: there is no forecast and nothing to lag. Its friction is booked once a year in January (a cost convention, not a lag), on the full notional of both legs — 6.0 bp/yr against a true drift-rebalancing cost of 0.03–0.29 bp/yr, i.e. charged 20–200× over.
2. **Frequency.** Monthly. Daily total-return closes for two funds paying on different dates carry large transient noise (SPY−IVV: 2.20% daily TE vs 0.52% monthly). Daily is reported as a robustness check and agrees to 1 bp.
3. **Non-tape inputs, and the one that binds.** Expense ratios are an **assumption** (current published net ERs held constant) — and since every cheap leg's fee was *cut* during the sample, that assumption inflates every residual. **Both** legs are swept and the whole table is re-run under three fee schedules; the sign does not survive it, so only a magnitude is reported. Borrow is an **assumption**, swept 0→100 bp.

> 💡 *In plain words:* take the return gap between two near-identical funds, add back the fee gap, and see whether anything is left over.

In [1]:
R = {'fp': 'd080dc025da8', 'asof': '2026-06-30', 'n_days_frame': 8411, 'pairs': [('SPY', 'IVV', 'US large cap', '2000-06', '2026-06', 313, -1.4, 6.5, 5.1, 1.04, -5.3, 15.1, 0.52, 20, 1.002), ('EEM', 'IEMG', 'Emerging markets', '2012-11', '2026-06', 164, -58.0, 61.0, 3.0, 0.11, -50.1, 56.2, 1.24, 67, 1.015), ('EFA', 'IEFA', 'Developed ex-US', '2012-11', '2026-06', 164, -21.4, 26.0, 4.6, 0.24, -33.6, 43.4, 0.82, 44, 0.993), ('VEA', 'IEFA', 'Developed ex-US (cross-sponsor)', '2012-11', '2026-06', 164, 52.7, -4.0, 48.7, 1.14, -25.1, 137.0, 1.26, 68, 1.022), ('IWM', 'IJR', 'US small cap', '2000-06', '2026-06', 313, -127.3, 13.0, -114.3, -1.5, -264.3, 30.0, 3.75, 147, 1.013)], 'pooled_resid': 3.7, 'pooled_t': 0.31, 'pooled_lo': -20.0, 'pooled_hi': 27.2, 'pooled_te': 0.55, 'pooled_floor': 30, 'pooled_n': 164, 'pooled_early': -0.6, 'pooled_early_t': -0.05, 'pooled_all': 2.1, 'pooled_all_t': 0.08, 'spy_bound': 5.3, 'spy_ci_bound': 15.1, 'era': [('SPY-IVV', 5.6, 0.9, 4.1, 0.64), ('EEM-IEMG', 14.9, 0.58, -6.8, -0.13), ('EFA-IEFA', -23.3, -0.89, 36.7, 1.34), ('VEA-IEFA', -16.2, -0.67, 125.8, 1.55), ('IWM-IJR', -159.3, -2.06, 6.5, 0.03)], 'er_sweep': [(0.68, 59.0, 1.0, 0.04), (0.7, 61.0, 3.0, 0.11), (0.72, 63.0, 5.0, 0.18), (0.75, 66.0, 8.0, 0.29)], 'er_sweep_cheap': [('SPY-IVV', 'IVV', [('earliest', 0.0945, 0.0, -1.4, -0.28), ('mid', 0.065, 2.9, 1.6, 0.32), ('today', 0.03, 6.5, 5.1, 1.04)]), ('EEM-IEMG', 'IEMG', [('earliest', 0.18, 52.0, -6.0, -0.22), ('mid', 0.13, 57.0, -1.0, -0.03), ('today', 0.09, 61.0, 3.0, 0.11)]), ('EFA-IEFA', 'IEFA', [('earliest', 0.14, 19.0, -2.4, -0.13), ('mid', 0.09, 24.0, 2.6, 0.13), ('today', 0.07, 26.0, 4.6, 0.24)])], 'fee_scenarios': [('today (headline)', [5.1, 3.0, 4.6, 48.7, -114.3]), ('approx time-weighted', [2.1, 0.0, 3.1, 49.7, -122.8]), ('earliest in-sample', [1.2, -1.0, -0.4, 50.7, -127.3])], 'turnover': [('IEMG/EEM', 3.36, 0.1), ('IEFA/EFA', 2.28, 0.07), ('IVV/SPY', 1.0, 0.03), ('IJR/IWM', 9.49, 0.28)], 'charged_bp': 6.0, 'daily': [('SPY-IVV', 4.0, 0.29, 2.2), ('EEM-IEMG', 4.0, 0.12, 1.75), ('EFA-IEFA', 4.0, 0.18, 1.32), ('VEA-IEFA', 47.7, 1.33, 1.83), ('IWM-IJR', -114.3, -1.49, 5.13)], 'trade': [('IEMG/EEM', 47.0, 0.38, 1.69, 22.0, -3.0, -53.0, 1.24), ('IEFA/EFA', 16.5, 0.2, 0.85, -8.5, -33.5, -83.5, 0.82), ('IVV/SPY', -5.2, -0.1, -1.02, -30.2, -55.2, -105.2, 0.53), ('IJR/IWM', 110.4, 0.29, 1.45, 85.4, 60.4, 10.4, 3.78)], 'ex_sharpe': [('EEM', 0.26), ('IEMG', 0.38), ('IWM', 0.43), ('IJR', 0.47), ('SPY', 0.64), ('IVV', 0.64)], 'syn_planted': 60.0, 'syn_recovered': 63.1, 'syn_t': 4.89, 'syn_null_mean': 2.7, 'syn_null_sd': 13.9, 'syn_null_fire': 0, 'syn_pool': 64.7, 'syn_pool_t': 10.37}

## The five pairs

In [2]:
hdr = ('pair       class                     window            n   drift    dER   '
       'resid      t        95% CI           TE    floor   beta')
print(hdr); print('-' * len(hdr))
for a, b, k, s, e, n, dr, der, res, t, lo, hi, te, fl, beta in R['pairs']:
    print('%-4s-%-5s %-25s %s->%s %4d %+7.1f %+6.1f %+8.1f %+6.2f '
          '[%+7.1f,%+7.1f] %5.2f%% %4dbp %6.3f'
          % (a, b, k[:25], s, e, n, dr, der, res, t, lo, hi, te, fl, beta))
print()
print('pooled (tight pairs, equal weight, n=%d): %+.1f bp/yr  t=%+.2f  '
      'CI[%+.1f,%+.1f]  TE %.2f%%  floor %dbp'
      % (R['pooled_n'], R['pooled_resid'], R['pooled_t'], R['pooled_lo'],
         R['pooled_hi'], R['pooled_te'], R['pooled_floor']))

pair       class                     window            n   drift    dER   resid      t        95% CI           TE    floor   beta
---------------------------------------------------------------------------------------------------------------------------------
SPY -IVV   US large cap              2000-06->2026-06  313    -1.4   +6.5     +5.1  +1.04 [   -5.3,  +15.1]  0.52%   20bp  1.002
EEM -IEMG  Emerging markets          2012-11->2026-06  164   -58.0  +61.0     +3.0  +0.11 [  -50.1,  +56.2]  1.24%   67bp  1.015
EFA -IEFA  Developed ex-US           2012-11->2026-06  164   -21.4  +26.0     +4.6  +0.24 [  -33.6,  +43.4]  0.82%   44bp  0.993
VEA -IEFA  Developed ex-US (cross-sp 2012-11->2026-06  164   +52.7   -4.0    +48.7  +1.14 [  -25.1, +137.0]  1.26%   68bp  1.022
IWM -IJR   US small cap              2000-06->2026-06  313  -127.3  +13.0   -114.3  -1.50 [ -264.3,  +30.0]  3.75%  147bp  1.013

pooled (tight pairs, equal weight, n=164): +3.7 bp/yr  t=+0.31  CI[-20.0,+27.2]  TE 0.55%  flo

**Reading it.** Betas within 1.00 ± 0.03 confirm these are same-exposure pairs. The three same-sponsor pairs return residuals of 3–5 bp/yr with |*t*| ≤ 1.04. The two cross-family pairs (VEA−IEFA, IWM−IJR) return large residuals that are composition wedges: IWM−IJR's −114 bp/yr against a 13 bp fee gap is the Russell-2000 / S&P-600 quality differential, and it does not clear |*t*| = 2 either.

> 💡 *In plain words:* the funds that really track the same thing show nothing; the funds that show something do not really track the same thing.

## The structural control — SPY cannot lend

SPY is a unit investment trust (1993 vintage): it may not lend securities and may not reinvest dividends between distributions. IVV is an open-end fund that lends. Identical index, 0.52%/yr relative tracking error — the tightest pair available anywhere, and 26 years of it.

A lending offset in IVV predicts a **negative** SPY−IVV residual (SPY should lose *more* than its fee handicap). The tape's own contribution is a realised drift of **−1.4 bp/yr**; everything after that is the fee assumption.

**Two confounds are baked in and are not separable**, so this control bounds a *net* structural difference rather than IVV's lending revenue: SPY's UIT structure also forbids reinvesting dividends between distributions (a cash drag penalising SPY), and SPY's own fee never moved while IVV's fell fourfold.

In [3]:
a, b, k, s, e, n, dr, der, res, t, lo, hi, te, fl, beta = R['pairs'][0]
early = R['er_sweep_cheap'][0][2][0]
print('SPY - IVV   %s -> %s   n=%d months   TE %.2f%%/yr   beta %.3f'
      % (s, e, n, te, beta))
print('  realised drift (THE TAPE)        %+.1f bp/yr' % dr)
print('  + fee gap, today\'s ERs   %+6.1f -> residual %+.1f bp/yr  HAC t %+.2f'
      % (der, res, t))
print('  + fee gap, 2000-era ERs  %+6.1f -> residual %+.1f bp/yr  HAC t %+.2f'
      % (early[2], early[3], early[4]))
print('  95%% CI on the headline [%+.1f, %+.1f] bp/yr' % (lo, hi))
print()
print('  -> the SIGN is the fee assumption\'s, not the tape\'s. What the tape')
print('     delivers is a BOUND: any IVV passthrough visible in returns is')
print('     <= %.0f bp/yr at 95%% confidence.' % hi)
print('  -> and it is a bound on a NET quantity: SPY\'s UIT dividend cash drag')
print('     pushes the same residual the other way and cannot be separated out.')

SPY - IVV   2000-06 -> 2026-06   n=313 months   TE 0.52%/yr   beta 1.002
  realised drift (THE TAPE)        -1.4 bp/yr
  + fee gap, today's ERs     +6.5 -> residual +5.1 bp/yr  HAC t +1.04
  + fee gap, 2000-era ERs    +0.0 -> residual -1.4 bp/yr  HAC t -0.28
  95% CI on the headline [-5.3, +15.1] bp/yr

  -> the SIGN is the fee assumption's, not the tape's. What the tape
     delivers is a BOUND: any IVV passthrough visible in returns is
     <= 15 bp/yr at 95% confidence.
  -> and it is a bound on a NET quantity: SPY's UIT dividend cash drag
     pushes the same residual the other way and cannot be separated out.


## The fee-history sweep — the assumption that owns the sign

The residual is a difference of two fees plus a drift, and fees are **not tape**. Every cheap leg here was cut hard during the sample (IVV 0.0945→0.03, IEMG 0.18→0.09, IEFA 0.14→0.07, IJR 0.20→0.06) while the dear legs barely moved. Holding today's ERs constant therefore overstates every fee gap and biases every residual **upward** — the direction that makes an offset look absent. Sweeping only the dear leg would be a one-sided sweep, so both legs are swept and the whole table is re-run under three labelled schedules.

*(The endpoints are published headline ERs; the mid value is an approximate time-weighting of the published cut schedule — an ASSUMPTION, bracketed by the endpoints, so nothing rests on it being exact.)*

In [4]:
print('CHEAP-leg sweep (dear leg pinned at today\'s ER):')
print('pair       cheap  assumption   ER_b      dER      residual      t')
for name, cheap, sweep in R['er_sweep_cheap']:
    for label, er_b, der_, res_, t_ in sweep:
        print('%-10s %-6s %-10s %.4f%%  %+7.1f bp  %+7.1f bp  %+6.2f'
              % (name, cheap, label, er_b, der_, res_, t_))
print()
print('WHOLE TABLE re-run, both legs moved:')
print('%-22s %8s %9s %9s %9s %9s' % ('scenario', 'SPY-IVV', 'EEM-IEMG',
                                     'EFA-IEFA', 'VEA-IEFA', 'IWM-IJR'))
for label, vals in R['fee_scenarios']:
    print('%-22s ' % label + ' '.join('%+8.1f' % v for v in vals))
print()
print('pooled (tight pairs): %+.1f bp (t %+.2f) on today\'s fees, %+.1f bp (t %+.2f)'
      % (R['pooled_resid'], R['pooled_t'], R['pooled_early'], R['pooled_early_t']))
print('pooled (ALL FIVE, today\'s fees): %+.1f bp (t %+.2f) - the class filter is'
      % (R['pooled_all'], R['pooled_all_t']))
print('shown rather than assumed: dropping the two wedge pairs changes nothing.')
print()
print('=> the tight-pair residuals are SMALLER than the uncertainty in the fee')
print('   input. Magnitude survives (|resid| <= ~6 bp/yr); the sign does not.')

CHEAP-leg sweep (dear leg pinned at today's ER):
pair       cheap  assumption   ER_b      dER      residual      t
SPY-IVV    IVV    earliest   0.0945%     +0.0 bp     -1.4 bp   -0.28
SPY-IVV    IVV    mid        0.0650%     +2.9 bp     +1.6 bp   +0.32
SPY-IVV    IVV    today      0.0300%     +6.5 bp     +5.1 bp   +1.04
EEM-IEMG   IEMG   earliest   0.1800%    +52.0 bp     -6.0 bp   -0.22
EEM-IEMG   IEMG   mid        0.1300%    +57.0 bp     -1.0 bp   -0.03
EEM-IEMG   IEMG   today      0.0900%    +61.0 bp     +3.0 bp   +0.11
EFA-IEFA   IEFA   earliest   0.1400%    +19.0 bp     -2.4 bp   -0.13
EFA-IEFA   IEFA   mid        0.0900%    +24.0 bp     +2.6 bp   +0.13
EFA-IEFA   IEFA   today      0.0700%    +26.0 bp     +4.6 bp   +0.24

WHOLE TABLE re-run, both legs moved:
scenario                SPY-IVV  EEM-IEMG  EFA-IEFA  VEA-IEFA   IWM-IJR
today (headline)           +5.1     +3.0     +4.6    +48.7   -114.3
approx time-weighted       +2.1     +0.0     +3.1    +49.7   -122.8
earliest in-sample

## Era cut (split 2020-01-01)

In [5]:
print('pair        early resid      t     |  late resid       t')
for name, er, et, lr, lt in R['era']:
    print('%-10s %+10.1f bp %+6.2f  | %+10.1f bp %+6.2f' % (name, er, et, lr, lt))
print()
print('Only SPY-IVV is stable across eras - and it is stable at zero.')
print('The only |t|>2 among the RESIDUAL estimates (IWM-IJR early) flips sign')
print('after 2020: an index-methodology fact, not a fee or lending fact - and it')
print('points the opposite way to the lending story anyway.')

pair        early resid      t     |  late resid       t
SPY-IVV          +5.6 bp  +0.90  |       +4.1 bp  +0.64
EEM-IEMG        +14.9 bp  +0.58  |       -6.8 bp  -0.13
EFA-IEFA        -23.3 bp  -0.89  |      +36.7 bp  +1.34
VEA-IEFA        -16.2 bp  -0.67  |     +125.8 bp  +1.55
IWM-IJR        -159.3 bp  -2.06  |       +6.5 bp  +0.03

Only SPY-IVV is stable across eras - and it is stable at zero.
The only |t|>2 among the RESIDUAL estimates (IWM-IJR early) flips sign
after 2020: an index-methodology fact, not a fee or lending fact - and it
points the opposite way to the lending story anyway.


## The expense-ratio sweep, dear leg

The smaller half of the assumption, for completeness: EEM's ER was 0.75% at IEMG's launch and is 0.70% now. The residual moves one-for-one with the assumed fee gap here too, but the dear legs moved far less than the cheap ones, so this band leaves the magnitude untouched.

In [6]:
print('EEM-IEMG, sweeping the assumed EEM expense ratio:')
for er, der, res, t in R['er_sweep']:
    print('  ER_EEM = %.2f%%  ->  dER %+.1f bp  residual %+.1f bp/yr  t = %+.2f'
          % (er, der, res, t))
print('\nacross the full band the residual stays within +-10 bp/yr and |t| < 0.3')

EEM-IEMG, sweeping the assumed EEM expense ratio:
  ER_EEM = 0.68%  ->  dER +59.0 bp  residual +1.0 bp/yr  t = +0.04
  ER_EEM = 0.70%  ->  dER +61.0 bp  residual +3.0 bp/yr  t = +0.11
  ER_EEM = 0.72%  ->  dER +63.0 bp  residual +5.0 bp/yr  t = +0.18
  ER_EEM = 0.75%  ->  dER +66.0 bp  residual +8.0 bp/yr  t = +0.29

across the full band the residual stays within +-10 bp/yr and |t| < 0.3


## Frequency robustness — daily instead of monthly

In [7]:
print('pair        daily resid       t     daily TE   (monthly TE)')
mte = {('%s-%s' % (p[0], p[1])): p[12] for p in R['pairs']}
for name, res, t, te in R['daily']:
    print('%-10s %+10.1f bp %+6.2f   %5.2f%%      %5.2f%%'
          % (name, res, t, te, mte[name]))
print('\nsame answers; daily TE is 2-4x larger purely from dividend-date mismatch,')
print('which is why the monthly series carries the headline.')

pair        daily resid       t     daily TE   (monthly TE)
SPY-IVV          +4.0 bp  +0.29    2.20%       0.52%
EEM-IEMG         +4.0 bp  +0.12    1.75%       1.24%
EFA-IEFA         +4.0 bp  +0.18    1.32%       0.82%
VEA-IEFA        +47.7 bp  +1.33    1.83%       1.26%
IWM-IJR        -114.3 bp  -1.49    5.13%       3.75%

same answers; daily TE is 2-4x larger purely from dividend-date mismatch,
which is why the monthly series carries the headline.


## The power arithmetic — why this tape can only bound

The noise floor is $2\,\sigma_{TE}/\sqrt{T}$: the smallest annual effect the sample can call real at |*t*| = 2. Set it beside a realistic net lending yield for a broad index fund (1–15 bp/yr, per Blocher & Whaley 2016).

In [8]:
print('pair        TE/yr   years   noise floor   plausible lending yield')
for a, b, k, s, e, n, dr, der, res, t, lo, hi, te, fl, beta in R['pairs']:
    print('%-4s-%-5s %5.2f%%  %5.1f   %6d bp    1-15 bp' % (a, b, te, n / 12.0, fl))
print('\npooled: floor %d bp/yr against a 1-15 bp effect -> the study is'
      % R['pooled_floor'])
print('underpowered BY CONSTRUCTION. The reportable quantity is the CI edge,')
print('not the point estimate: a bound, not a measurement.')

pair        TE/yr   years   noise floor   plausible lending yield
SPY -IVV    0.52%   26.1       20 bp    1-15 bp
EEM -IEMG   1.24%   13.7       67 bp    1-15 bp
EFA -IEFA   0.82%   13.7       44 bp    1-15 bp
VEA -IEFA   1.26%   13.7       68 bp    1-15 bp
IWM -IJR    3.75%   26.1      147 bp    1-15 bp

pooled: floor 30 bp/yr against a 1-15 bp effect -> the study is
underpowered BY CONSTRUCTION. The reportable quantity is the CI edge,
not the point estimate: a bound, not a measurement.


## The tradable leg — long cheap / short dear, with borrow

Dollar-neutral and **static**: two fixed tickers, no signal, so no execution lag is applied and none is claimed. The arithmetic is a book restored to equal notional each month end; friction is charged once a year on the **full** notional of both legs (6.0 bp/yr, booked in January — a cost convention). That charge is deliberately conservative, as the turnover check below shows. Self-financing, so the return is already excess-of-cash and the Sharpe is comparable to any other excess-of-cash Sharpe on the desk — with the standing caveat that a real short rebate lands *below* the cash rate, which is what the borrow sweep is for.

In [9]:
print('cost conservatism: true drift-rebalancing turnover vs what is charged')
for name, turn, true_bp in R['turnover']:
    print('  %-10s turnover %5.2f%%/yr -> %.2f bp/yr true  vs %.1f bp/yr charged'
          % (name, turn, true_bp, R['charged_bp']))
print('  (charged 20-200x over: the trade is not flattered by its cost model)')
print()
print('trade       vol     0bp borrow          25bp      50bp      100bp')
for name, g0, sh, t, b25, b50, b100, vol in R['trade']:
    print('%-10s %5.2f%%  %+7.1f bp (Sh %+.2f, t %+.2f)  %+7.1f  %+7.1f  %+7.1f'
          % (name, vol, g0, sh, t, b25, b50, b100))
print()
print('IEMG/EEM harvests the 61 bp FEE gap, not lending revenue - and it is')
print('exactly zero at 50 bp of borrow, inside the range EM ETFs borrow at.')
print('IJR/IWM +110 bp is the small-cap index quality premium (TE 3.78%, t +1.45),')
print('a different study entirely.')
print()
print('context - each leg excess-of-cash Sharpe vs BIL:')
print('  ' + '  '.join('%s %+.2f' % (k, v) for k, v in R['ex_sharpe']))

cost conservatism: true drift-rebalancing turnover vs what is charged
  IEMG/EEM   turnover  3.36%/yr -> 0.10 bp/yr true  vs 6.0 bp/yr charged
  IEFA/EFA   turnover  2.28%/yr -> 0.07 bp/yr true  vs 6.0 bp/yr charged
  IVV/SPY    turnover  1.00%/yr -> 0.03 bp/yr true  vs 6.0 bp/yr charged
  IJR/IWM    turnover  9.49%/yr -> 0.28 bp/yr true  vs 6.0 bp/yr charged
  (charged 20-200x over: the trade is not flattered by its cost model)

trade       vol     0bp borrow          25bp      50bp      100bp
IEMG/EEM    1.24%    +47.0 bp (Sh +0.38, t +1.69)    +22.0     -3.0    -53.0
IEFA/EFA    0.82%    +16.5 bp (Sh +0.20, t +0.85)     -8.5    -33.5    -83.5
IVV/SPY     0.53%     -5.2 bp (Sh -0.10, t -1.02)    -30.2    -55.2   -105.2
IJR/IWM     3.78%   +110.4 bp (Sh +0.29, t +1.45)    +85.4    +60.4    +10.4

IEMG/EEM harvests the 61 bp FEE gap, not lending revenue - and it is
exactly zero at 50 bp of borrow, inside the range EM ETFs borrow at.
IJR/IWM +110 bp is the small-cap index quality premiu

## Live synthetic control — the estimator is unbiased and powered

Planted world: the dearer fund is handed 60 bp/yr of passthrough on top of its fee; the estimator must recover it. Null world: fees and composition noise only; the estimator must stay quiet. This is simulated data — it never supports the real-tape stamp, it only certifies the harness.

In [10]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..', '..', '..')))
import numpy as np
from lending_offset import data, strategy as st
pl = st.synthetic_detect(*data.synthetic_daily(signal_strength=1.0, seed=914))
print('planted %+.0f bp/yr -> recovered %+.1f bp/yr (error %+.1f)  t=%+.2f  TE %.2f%%'
      % (pl['planted_bp'], pl['residual_bp'], pl['error_bp'], pl['t_hac'], pl['te_pct']))
half = st.synthetic_detect(*data.synthetic_daily(signal_strength=0.5, seed=914))
print('half strength %+.0f bp -> %+.1f bp  t=%+.2f  (monotone in the knob)'
      % (half['planted_bp'], half['residual_bp'], half['t_hac']))
nulls = [st.synthetic_detect(*data.synthetic_daily(signal_strength=0.0, seed=914 + s))
         for s in range(8)]
res = np.array([n['residual_bp'] for n in nulls])
ts  = np.array([n['t_hac'] for n in nulls])
print('null x8: mean %+.1f bp (sd %.1f), |t|>=2 in %d/8'
      % (res.mean(), res.std(ddof=1), (abs(ts) >= 2).sum()))
frames, tr = data.synthetic_panel(signal_strength=1.0, seed=914)
pool = st.synthetic_panel_detect(frames, tr)
print('pooled %d planted pairs: %+.1f bp  t=%+.2f  (pooling buys power, as on the desk)'
      % (pool['n_pairs'], pool['residual_bp'], pool['t_hac']))

planted +60 bp/yr -> recovered +63.1 bp/yr (error +3.1)  t=+4.89  TE 0.70%


half strength +30 bp -> +32.0 bp  t=+2.49  (monotone in the knob)


null x8: mean +2.7 bp (sd 13.9), |t|>=2 in 0/8


pooled 5 planted pairs: +64.7 bp  t=+10.37  (pooling buys power, as on the desk)


## Verdict

- **Signal — None.** The fee-adjusted residual on the three same-sponsor pairs is **+3.0 / +4.6 / +5.1 bp/yr**, |*t*| ≤ 1.04, every CI straddling zero; pooled it is **+3.7 bp/yr (*t* = +0.31)**, CI [-20.0, +27.2] — and **-0.6 bp/yr (*t* = -0.05)** under earliest-in-sample fees. The structural control — SPY, a unit investment trust that legally **cannot** lend, against IVV, which does — shows a realised drift of only −1.4 bp/yr over 26 years, bounding any visible IVV passthrough at **≈15 bp/yr** (95% CI). The two large residuals (IWM−IJR −114 bp, VEA−IEFA +49 bp) are index-composition wedges and neither clears |*t*| = 2. The synthetic control recovers a planted +60 bp at *t* = +4.89 and is silent on the null (+2.7 bp, 0/8), so the harness is sound. Named limits: **power** (noise floor 20–147 bp/yr against a 1–15 bp effect); **the fee assumption owns the sign**, so a magnitude is reported and no direction; the SPY/IVV control **confounds** lending with SPY's UIT dividend cash drag; and **fund survivorship** (all ten funds are survivors).
- **Tradability — Mirage.** No lending spread exists to bank. The fee gap is tradable at +47 bp/yr gross (IEMG/EEM) and is **exactly zero at 50 bp of borrow** — inside the realistic range. Own the cheap fund; do not short the dear one.
- **The deliverable is a magnitude bound — not an estimate, and not a sign.** Whatever the lending desks earn, what reaches the shareholder as measurable relative return is under ≈6 bp/yr on every same-sponsor pair under every fee scenario (CI edge ≈15 bp/yr on the tightest pair, ≈25 bp/yr pooled). Real in the annual report; at its plausible size, invisible in this tape — which was never capable of resolving it either way.